In [1]:
!pip install -q unsloth transformers datasets accelerate sentencepiece bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
max_seq_length = 1024

In [6]:
from transformers import AutoTokenizer # Keep AutoTokenizer for potential separate loading or general use
from unsloth import FastLanguageModel
import torch

# CONFIG
MODEL_PATH = "/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/merged_model"
EVAL_DATASET_PATH = "/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/eval_dataset.jsonl"

# LOAD MODEL
# Use Unsloth's FastLanguageModel.from_pretrained for correct patching
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=max_seq_length, # Use the previously defined max_seq_length
    load_in_4bit=True,
)

model.eval()

print("Model device:", next(model.parameters()).device)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1531: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model device: cuda:0


In [8]:
# generate synthesised dataset to evaluate the model

import sys
sys.path.append('/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml')
from synth_pipeline_v5_2 import generate_synth_data
from datasets import Dataset
import json

# Your existing structured generation setup
NUM_SAMPLES = 300
data = []

EOS_TOKEN = tokenizer.eos_token # Essential to avoid infinite loops during inference

for _ in range(NUM_SAMPLES):
    example = generate_synth_data()
    text_structure = f"""### Instruction:
{example["instruction"]}

### Schema:
{json.dumps(example["schema"], ensure_ascii=False, indent=2)}

### Input:
{example["input"]}

### Response:
{json.dumps(example["output"], ensure_ascii=False, indent=2)}{EOS_TOKEN}"""

    data.append({"text": text_structure})

dataset = Dataset.from_list(data)
# dataset = dataset.train_test_split(test_size=0.1, seed=42)

# train_dataset = dataset["train"]
# eval_dataset = dataset["test"]
eval_dataset = dataset


In [9]:
# ============================================
# RUN MODEL ON EVALUATION DATASET
# ============================================

import json
import torch

predictions = []
expected_outputs = []

for i, sample in enumerate(eval_dataset):

    full_text = sample["text"]

    # Everything before ### Response is the model input
    prompt = full_text.split("### Response:")[0] + "### Response:\n"

    # Ground truth
    expected = (
        full_text.split("### Response:", 1)[1]
        .replace(tokenizer.eos_token, "")
        .strip()
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        # max_length=max_seq_length,
        # max_length=512,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    prediction = generated[len(prompt):].strip()

    predictions.append(prediction)
    expected_outputs.append(expected)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(eval_dataset)}")

Processed 10/300
Processed 20/300
Processed 30/300
Processed 40/300
Processed 50/300
Processed 60/300
Processed 70/300
Processed 80/300
Processed 90/300
Processed 100/300
Processed 110/300
Processed 120/300
Processed 130/300
Processed 140/300
Processed 150/300
Processed 160/300
Processed 170/300
Processed 180/300
Processed 190/300
Processed 200/300
Processed 210/300
Processed 220/300
Processed 230/300
Processed 240/300
Processed 250/300
Processed 260/300
Processed 270/300
Processed 280/300
Processed 290/300
Processed 300/300


In [10]:
# ============================================
# INSPECT RESULTS
# ============================================

for i in range(min(5, len(predictions))):

    print("=" * 80)
    print(f"SAMPLE {i}")
    print("=" * 80)

    print("\nEXPECTED:")
    print(expected_outputs[i])

    print("\nPREDICTED:")
    print(predictions[i])

    print()

SAMPLE 0

EXPECTED:
{
  "person": {
    "givenName": "Ama"
  },
  "bioData": {
    "surname": "Gyamfi",
    "date_of_birth": null
  },
  "contactDetails": {
    "secondaryPhone": null
  },
  "employerDetails": {
    "employeeNumber": null
  },
  "nextOfKin": {
    "relationship": null
  },
  "emergencyContact": {
    "name": null
  },
  "identifiers": {
    "nationalId": null
  }
}

PREDICTED:
{
  "person": {
    "givenName": "Ama"
  },
  "bioData": {
    "surname": "Gyamfi",
    "date_of_birth": null
  },
  "contactDetails": {
    "secondaryPhone": null
  },
  "employerDetails": {
    "employeeNumber": null
  },
  "nextOfKin": {
    "relationship": null
  },
  "emergencyContact": {
    "name": null
  },
  "identifiers": {
    "nationalId": null
  }
}

SAMPLE 1

EXPECTED:
{
  "firstName": "Adwoa Mansa",
  "surname": "Renner",
  "fullName": null,
  "ssnit_number": null,
  "fromTrustee": "Metropolitan Pensions Trust Ghana Limited"
}

PREDICTED:
{
  "firstName": "Adwoa Mansa",
  "surname"

In [11]:
# ============================================
# PARSE PREDICTIONS
# ============================================

parsed_predictions = []
json_valid = []

for prediction in predictions:

    try:
        parsed = json.loads(prediction)

        parsed_predictions.append(parsed)
        json_valid.append(True)

    except json.JSONDecodeError:
        parsed_predictions.append(None)
        json_valid.append(False)

print(
    f"Valid JSON: {sum(json_valid)}/{len(json_valid)} "
    f"({sum(json_valid) / len(json_valid) * 100:.2f}%)"
)

Valid JSON: 300/300 (100.00%)


In [12]:
# ============================================
# FIELD-LEVEL EVALUATION
# ============================================

field_correct = {}
field_total = {}

total_correct = 0
total_fields = 0

for expected_text, predicted in zip(expected_outputs, parsed_predictions):

    # Parse expected JSON
    try:
        expected = json.loads(expected_text)
    except json.JSONDecodeError:
        continue

    # If prediction isn't valid JSON
    if predicted is None:
        for field in expected:
            field_total[field] = field_total.get(field, 0) + 1
            total_fields += 1

        continue

    # Compare every expected field
    for field, expected_value in expected.items():

        field_total[field] = field_total.get(field, 0) + 1
        total_fields += 1

        predicted_value = predicted.get(field)

        if predicted_value == expected_value:
            field_correct[field] = field_correct.get(field, 0) + 1
            total_correct += 1

print("\nFIELD-LEVEL RESULTS")
print("=" * 60)

for field in field_total:

    accuracy = (
        field_correct.get(field, 0)
        / field_total[field]
        * 100
    )

    print(f"{field:30s}: {accuracy:.2f}%")

overall_accuracy = total_correct / total_fields * 100

print("\n" + "=" * 60)
print(f"OVERALL FIELD ACCURACY: {overall_accuracy:.2f}%")


FIELD-LEVEL RESULTS
person                        : 100.00%
bioData                       : 100.00%
contactDetails                : 100.00%
employerDetails               : 100.00%
nextOfKin                     : 100.00%
emergencyContact              : 100.00%
identifiers                   : 100.00%
firstName                     : 96.00%
surname                       : 96.55%
fullName                      : 100.00%
ssnit_number                  : 100.00%
fromTrustee                   : 100.00%
member                        : 97.01%
identification                : 100.00%
contact                       : 98.59%
employment                    : 98.41%
pension                       : 97.80%
givenName                     : 100.00%
last_name                     : 100.00%
monthlySalary                 : 100.00%
staffNumber                   : 100.00%
sex                           : 100.00%
contributions                 : 100.00%
individual                    : 91.43%
ids                       

In [13]:
# ============================================
# EXACT RECORD ACCURACY
# ============================================

exact_matches = 0
valid_expected = 0

for expected_text, predicted in zip(expected_outputs, parsed_predictions):

    try:
        expected = json.loads(expected_text)
    except json.JSONDecodeError:
        continue

    valid_expected += 1

    if predicted == expected:
        exact_matches += 1

exact_accuracy = exact_matches / valid_expected * 100

print(f"Exact record matches: {exact_matches}/{valid_expected}")
print(f"Exact record accuracy: {exact_accuracy:.2f}%")

Exact record matches: 277/300
Exact record accuracy: 92.33%


In [14]:
# ============================================
# SCHEMA COMPLIANCE
# ============================================

schema_compliant = 0
schema_total = len(parsed_predictions)

for expected_text, predicted in zip(expected_outputs, parsed_predictions):

    if predicted is None:
        continue

    expected = json.loads(expected_text)

    expected_keys = set(expected.keys())
    predicted_keys = set(predicted.keys())

    if expected_keys == predicted_keys:
        schema_compliant += 1

schema_accuracy = schema_compliant / schema_total * 100

print(f"Schema-compliant records: {schema_compliant}/{schema_total}")
print(f"Schema compliance: {schema_accuracy:.2f}%")

Schema-compliant records: 299/300
Schema compliance: 99.67%


In [15]:
# ============================================
# FINAL EVALUATION SUMMARY
# ============================================

json_validity = sum(json_valid) / len(json_valid) * 100

print("=" * 70)
print("QWEN2.5-1.5B EXTRACTION EVALUATION")
print("=" * 70)

print(f"Evaluation records:       {len(eval_dataset)}")
print(f"JSON validity:            {json_validity:.2f}%")
print(f"Field-level accuracy:     {overall_accuracy:.2f}%")
print(f"Exact-record accuracy:    {exact_accuracy:.2f}%")
print(f"Schema compliance:        {schema_accuracy:.2f}%")

print("=" * 70)

QWEN2.5-1.5B EXTRACTION EVALUATION
Evaluation records:       300
JSON validity:            100.00%
Field-level accuracy:     98.43%
Exact-record accuracy:    92.33%
Schema compliance:        99.67%
